In [24]:
%pip install pandas
%pip install alpaca-py
%pip install pytz


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [25]:
import pandas as pd

In [ ]:
import pandas as pd
import numpy as np
#DATA gererated by AI TO CHECK THE CODE BELOW
np.random.seed(42)


tickers = {
    "AAA": 100,     
    "BBB": 50,      
    "CCC": 10,      
    "DDD": 200,     
    "EEE": 5        
}

start_date = pd.to_datetime("2025-01-02")
num_days = 3
minutes_per_day = 30  # simplified intraday

rows = []

for ticker, base_price in tickers.items():
    for d in range(num_days):
        date = start_date + pd.Timedelta(days=d)

        price = base_price

        for m in range(minutes_per_day):
            timestamp = date + pd.Timedelta(minutes=570 + m)  # 9:30am start

            # simulate small price movement
            change = np.random.normal(0, 0.2)
            open_price = price
            close_price = price + change
            high_price = max(open_price, close_price) + abs(np.random.normal(0, 0.1))
            low_price = min(open_price, close_price) - abs(np.random.normal(0, 0.1))

            # simulate different liquidity levels
            volume_multiplier = {
                "AAA": 5000,
                "BBB": 20000,
                "CCC": 800,
                "DDD": 40000,
                "EEE": 300
            }

            volume = int(abs(np.random.normal(volume_multiplier[ticker], volume_multiplier[ticker]*0.1)))

            rows.append([
                ticker,
                timestamp,
                round(open_price, 2),
                round(high_price, 2),
                round(low_price, 2),
                round(close_price, 2),
                volume
            ])

            price = close_price

columns = ["Ticker", "Date", "Open", "High", "Low", "Close", "Volume"]

df = pd.DataFrame(rows, columns=columns)

df.to_excel("test4stocks.xlsx", index=False)

print("Excel file 'test4stocks.xlsx' created successfully.")

Excel file 'test4stocks.xlsx' created successfully.


In [ ]:
import pandas as pd

# Load Excel
df = pd.read_excel("test4stocks.xlsx") #can be changed to whatever type we want 
df = df.dropna() # clean empty lines
df["Date"] = pd.to_datetime(df["Date"]) # converts data from the fprm of YYYY MM DD Time type to pandas  

final_set = [] # empty list for dollar bars


for ticker, ticker_df in df.groupby("Ticker"):  #split per tickers --- apple, nvda etc
    for date, day_df in ticker_df.groupby("Date"):  #similarly split by day
        day_df = day_df.sort_values("Date").reset_index(drop=True) # making sure the chronological order is fine and every piece starts as 0 index

    #Dollar Volume
        day_df["DollarVolume"] = day_df["Close"] * day_df["Volume"]

        # 50 bars per day target
        Strategist_Threshold = day_df["DollarVolume"].sum() / 50    #calculate the total dv per day and /50

        current_dollar_value = 0
        current_set = []

        #dollar bar
        for index, row in day_df.iterrows():

            dollar_tot = row["DollarVolume"]
            current_dollar_value += dollar_tot
            current_set.append(row)

            if current_dollar_value >= Strategist_Threshold:

                new_df_dollar_bar = pd.DataFrame(current_set) # new dataframe

                ticker_name = new_df_dollar_bar.iloc[0]["Ticker"]
                open_price = new_df_dollar_bar.iloc[0]["Close"]
                close_price = new_df_dollar_bar.iloc[-1]["Close"]
                high_price = new_df_dollar_bar["Close"].max()
                low_price = new_df_dollar_bar["Close"].min()

                final_set.append({
                    "Ticker": ticker_name,
                    "Date": date,
                    "Open": open_price,
                    "High": high_price,
                    "Low": low_price,
                    "Close": close_price,
                    "Dollar Volume": Strategist_Threshold
                })

           
                current_dollar_value -= Strategist_Threshold
                current_set = []

# Final DataFrame
final_set_df = pd.DataFrame(final_set)

print(final_set_df)

    Ticker                Date    Open    High     Low   Close  Dollar Volume
0      AAA 2025-01-02 09:30:00  100.10  100.10  100.10  100.10     11533.5220
1      AAA 2025-01-02 09:31:00  100.05  100.05  100.05  100.05     10771.3830
2      AAA 2025-01-02 09:32:00   99.96   99.96   99.96   99.96      9530.1864
3      AAA 2025-01-02 09:33:00  100.01  100.01  100.01  100.01      9436.9436
4      AAA 2025-01-02 09:34:00   99.80   99.80   99.80   99.80      8568.8280
..     ...                 ...     ...     ...     ...     ...            ...
445    EEE 2025-01-04 09:55:00    3.86    3.86    3.86    3.86        24.8584
446    EEE 2025-01-04 09:56:00    3.87    3.87    3.87    3.87        21.4398
447    EEE 2025-01-04 09:57:00    3.94    3.94    3.94    3.94        23.8764
448    EEE 2025-01-04 09:58:00    3.90    3.90    3.90    3.90        21.6060
449    EEE 2025-01-04 09:59:00    3.84    3.84    3.84    3.84        23.0400

[450 rows x 7 columns]
